In [3]:
from aiida import load_profile
load_profile()

from aiida.orm import Dict, load_node, load_code
from aiida.orm import StructureData, InstalledCode, Computer
from aiida.engine import submit
from aiida_kkr.workflows.kkr_scf import kkr_scf_wc
from aiida_kkr.workflows.kkr_bdg_wc import kkr_bdg_wc
import numpy as np

# struc = load_node('95fdec5c-17bf-4894-9867-d3031531e11e')
struc = StructureData(cell=np.array([[0.5, 0.5, -0.5], [0.5, -0.5, 0.5], [-0.5, 0.5, 0.5]])*3.3)
struc.append_atom(position=[0,0,0], symbols='Nb')
struc.label = 'Nb_bulk'
struc.description = 'bcc Nb bulk structure'


voro_code    = load_code('voronoi_3.5_AMD@iffslurm')         
kkr_code     = load_code('kkrhost_BdG_AMD@iffslurm')
kkr_bdg_code = load_code('kkrhost_BdG_AMD@iffslurm')


options = Dict(dict={
    'withmpi': True,
    'resources': {'num_machines': 1, 'tot_num_mpiprocs': 32},
    'queue_name': 'th1-2020-32',
    'max_wallclock_seconds': 3600 * 12,
})

calc_params = Dict(dict={
    'LMAX': 2,
    'NSPIN': 1,
    'RMAX': 10.0,
    'GMAX': 100.0,
})
# SCF workflow settings
scf_settings_dict = kkr_scf_wc.get_wf_defaults(silent=True)[0]
scf_settings_dict['mag_init']  = False
scf_settings_dict['check_dos'] = False
scf_settings_dict['nsteps']    = 100 
scf_settings_dict['convergence_setting_coarse'] = scf_settings_dict['convergence_setting_fine']
wf_params = Dict(dict=scf_settings_dict)
# Semi-circle settings
semi_circle_settings = Dict(dict={
    'USE_SEMI_CIRCLE_CONTOUR':   True,
    'IM_E_CIRC_MIN':             5e-5,
    'NPT1':                      32,
    'MAX_NUM_KMESH':             4,
    'BZDIVIDE':                  [100, 100, 100],
    'RCLUSTZ':                   3.5,
    'NSTEPS':                    200,
    'IMIX':                      4,
    'DISABLE_CHARGE_NEUTRALITY': True,
    'RUNOPT':                    ['NEWSOSOL'],
    'R_LOG':                     0.6,
    'NPAN_EQ':                   7,
    'NPAN_LOG':                  18,
    'NCHEB':                     12,
    'DECOUPLE_SPIN_CHEBY':       True,
})

# BdG settings 
bdg_init_settings = Dict(dict={
    'NSTEPS':              1,
    'use_BdG':             True,
    'Delta_BdG':           5e-4,
    'use_e_symm_BdG':      True,
    'at_scale_BdG':        [1.0 for _ in struc.sites],
})

bdg_settings = Dict(dict={
    'NSTEPS':                    200,
    'NSIMPLEMIXFIRST':           25,
    'at_scale_BdG':              [1.0],
    'lambda_BdG':                0.025,
    'mixfac_BdG':                0.3,
    'Ninit_Broyden_BdG':         5,
    'IMIX':                      4,
})


builder = kkr_bdg_wc.get_builder()
builder.structure            = struc
builder.voronoi              = voro_code
builder.kkr                  = kkr_code
builder.kkr_bdg              = kkr_bdg_code
builder.calc_parameters      = calc_params
builder.semi_circle_settings = semi_circle_settings
builder.bdg_init_settings    = bdg_init_settings
builder.bdg_settings         = bdg_settings
builder.options              = options
builder.wf_parameters        = wf_params

node = submit(builder)
print(f'Submitted: {node.pk}')